In [1]:
!pip install pandas googlesearch-python requests beautifulsoup4

In [2]:
import pandas as pd
from googlesearch import search
import requests
from bs4 import BeautifulSoup
import time
import urllib3


In [3]:
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [4]:
queries = [
    '"Genome Valley" Hyderabad "biotech" OR "manufacturing" company',
    '"Medical Devices Park" Hyderabad "medical device" OR "equipment" manufacturer',
    '"Jeedimetla" Hyderabad "specialty chemicals" OR "API" manufacturer',
    'Hyderabad "custom synthesis" "pharma" -distributor -retail'
]

raw_data = []
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

print("Initiating data extraction pipeline. This may take 3-5 minutes...\n")

for query in queries:
    print(f"Scraping results for: {query}")
    try:
        for url in search(query, num=15, stop=15, pause=3.0):
            if any(exclude in url.lower() for exclude in ['justdial', 'indiamart', 'zauba', 'tofler', 'linkedin', 'facebook', 'glassdoor']):
                continue
                
            company_name = "Unknown"
            try:
                response = requests.get(url, headers=headers, timeout=5, verify=False)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.text, 'html.parser')
                    if soup.title:
                        company_name = soup.title.text.strip().replace('\n', '').replace('\r', '')
            except Exception:
                company_name = url.split("//")[-1].split("/")[0].replace("www.", "")
            raw_data.append({
                "Source_Query": query,
                "Estimated_Company_Name": company_name[:75],
                "Website_URL": url,
                "E1_Producer_Pass": "",
                "E2_Accessible_Pass": "",
                "C3_Differentiated": "",
                "C4_Decision_Maker": "",
                "C6_Growth_Signals": "",
                "C7_Systems_Maturity": ""
            })
    except Exception as e:
        print(f"  -> Search error on '{query}': {e}")
df_raw = pd.DataFrame(raw_data).drop_duplicates(subset=['Website_URL'])

csv_filename = "hyderabad_target_companies_raw.csv"
df_raw.to_csv(csv_filename, index=False)

print(f"\nExtraction complete! {len(df_raw)} unique companies saved to {csv_filename}.")
df_raw.head()






Initiating data extraction pipeline. This may take 3-5 minutes...

Scraping results for: "Genome Valley" Hyderabad "biotech" OR "manufacturing" company
  -> Search error on '"Genome Valley" Hyderabad "biotech" OR "manufacturing" company': search() got an unexpected keyword argument 'num'
Scraping results for: "Medical Devices Park" Hyderabad "medical device" OR "equipment" manufacturer
  -> Search error on '"Medical Devices Park" Hyderabad "medical device" OR "equipment" manufacturer': search() got an unexpected keyword argument 'num'
Scraping results for: "Jeedimetla" Hyderabad "specialty chemicals" OR "API" manufacturer
  -> Search error on '"Jeedimetla" Hyderabad "specialty chemicals" OR "API" manufacturer': search() got an unexpected keyword argument 'num'
Scraping results for: Hyderabad "custom synthesis" "pharma" -distributor -retail
  -> Search error on 'Hyderabad "custom synthesis" "pharma" -distributor -retail': search() got an unexpected keyword argument 'num'

Extraction com

""
